# The Calculus of Backpropagation

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 02.09 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/10_calculus_of_backpropagation.ipynb)

---

## 🎯 Learning Objective

Derive backpropagation from scratch for dense, convolutional, and attention layers, and learn to diagnose vanishing/exploding gradients from the resulting gradient-flow equations.

---

## 📐 Theory

This capstone notebook synthesizes every idea from `02.01`–`02.09` into the one algorithm that
trains virtually every modern neural network. There is no new mathematical machinery here —
only the chain rule (`02.04`), applied layer by layer, with careful bookkeeping.

### Backprop through a dense layer

A dense layer computes $\mathbf{z} = W^\top\mathbf{x} + \mathbf{b}$ for $\mathbf{x}\in\mathbb{R}^n$,
$W\in\mathbb{R}^{n\times m}$, $\mathbf{b},\mathbf{z}\in\mathbb{R}^m$ — i.e. $W$'s rows index
the input and its columns index the output, matching `x @ W` for a NumPy row vector `x` (the
shape convention the code below actually uses; many textbooks instead write $W\in\mathbb{R}^{m\times n}$
and $\mathbf{z}=W\mathbf{x}$, which just transposes every formula below — the physics is
identical, but the two conventions are **not** interchangeable term-by-term, so we fix one and
stick to it). Then an elementwise activation $\mathbf{a} = \sigma(\mathbf{z})$. Given the
upstream gradient $\partial L/\partial\mathbf{a}$, the chain rule (in Jacobian form, from
`02.06`) gives:

$$\frac{\partial L}{\partial \mathbf{z}} = \frac{\partial L}{\partial \mathbf{a}} \odot \sigma'(\mathbf{z}), \quad
\frac{\partial L}{\partial W} = \mathbf{x}\,\frac{\partial L}{\partial \mathbf{z}}^\top, \quad
\frac{\partial L}{\partial \mathbf{b}} = \frac{\partial L}{\partial \mathbf{z}}, \quad
\frac{\partial L}{\partial \mathbf{x}} = W \frac{\partial L}{\partial \mathbf{z}}$$

($\odot$ is elementwise product.) $\partial L/\partial W$ is an outer product
($\mathbf{x}\in\mathbb{R}^n$ against $\partial L/\partial\mathbf{z}\in\mathbb{R}^m$, giving
the same $n\times m$ shape as $W$ — this is `np.outer(x, dL_dz)` below, *not* `np.outer(dL_dz, x)`,
since those two differ by a transpose and only one matches $W$'s actual shape); numerically,
$W\,\partial L/\partial\mathbf{z}$ and `dL_dz @ W.T` are the same vector. Each formula is the
local-gradient recipe from `02.04`: the activation's local gradient is $\sigma'(\mathbf{z})$;
the linear map's local gradient w.r.t. each of $W,\mathbf{b},\mathbf{x}$ follows from the
multivariate chain rule. Crucially, $\partial L/\partial\mathbf{x}$ is exactly the upstream
gradient the *previous* layer receives — this is what makes stacking layers work: each layer
only backprops through itself.

### Backprop through convolution and attention

A convolution is a dense layer with shared, sparse weights; its backward pass convolves the
upstream gradient with the *flipped* kernel to get the input gradient, and convolves the input
with the upstream gradient to get the kernel gradient — same local-gradient logic, sparser map.

Attention computes $\mathbf{o} = \text{softmax}(QK^\top/\sqrt{d_k})V$ (recall `01.10`).
Backprop here has three receiving ends ($Q$, $K$, $V$), but the mechanism is identical:
propagate the upstream gradient back through the product with $V$, then through softmax
(softmax is NOT elementwise, so its local gradient is a full matrix per row, not a vector),
then through $QK^\top$. Nothing new — the chain rule applied to more matrix products in sequence.

### Gradient flow: vanishing and exploding gradients

Chaining $n$ layers multiplies $n$ local Jacobians together (`02.04`'s reverse-mode recipe).
If each layer's local gradient has singular values consistently below $1$ (sigmoid's
derivative maxes out at $0.25$), the product shrinks geometrically with depth — **vanishing
gradients**: early layers get a signal too small to learn from. If local gradients are
consistently above $1$ (large weights, unsaturated ReLU), the product grows geometrically —
**exploding gradients**. Both are mechanical consequences of multiplying many numbers
repeatedly, nothing more. This explains why architectures use residual connections (an
identity path with local gradient exactly $1$, regardless of depth), why LayerNorm/BatchNorm
exist (control activation scale so local gradients stay well-behaved), and why non-saturating
activations like ReLU replaced sigmoid/tanh in deep networks.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Tracking gradient norm layer-by-layer as it backpropagates through a deep stack makes vanishing
and exploding gradients visible directly: a sigmoid stack's gradient norm collapses toward the
input layer, while an unnormalized ReLU stack's gradient norm can blow up instead.

In [ ]:
# Track ||gradient|| layer-by-layer as it backpropagates through a deep stack of random
# layers, for sigmoid (saturating) vs ReLU (non-saturating) activations.
rng = np.random.default_rng(0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_norms_through_depth(n_layers, activation, width=20, weight_scale=0.5):
    Ws = [rng.standard_normal((width, width)) * weight_scale for _ in range(n_layers)]
    x = rng.standard_normal(width)
    activs, zs = [x], []
    for W in Ws:
        z = activs[-1] @ W
        zs.append(z)
        activs.append(sigmoid(z) if activation == "sigmoid" else np.maximum(0, z))

    grad = np.ones(width)          # dL/d(final activation), for a toy loss L = sum(activation)
    norms = [np.linalg.norm(grad)]
    for i in reversed(range(n_layers)):
        local_grad = activs[i + 1] * (1 - activs[i + 1]) if activation == "sigmoid" else (zs[i] > 0)
        grad = (grad * local_grad) @ Ws[i].T   # exactly the dense-layer backward formula above
        norms.append(np.linalg.norm(grad))
    return norms[::-1]   # index 0 = gradient that reaches the INPUT layer

n_layers = 20
sigmoid_norms = gradient_norms_through_depth(n_layers, "sigmoid")
relu_norms = gradient_norms_through_depth(n_layers, "relu")

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(sigmoid_norms, 'o-', color="#C44E52", label="sigmoid (vanishing)")
ax.semilogy(relu_norms, 'o-', color="#4C72B0", label="ReLU, unnormalized weights (exploding)")
ax.set_xlabel("layer depth (0 = input side, 20 = output/loss side)")
ax.set_ylabel("||gradient|| (log scale)")
ax.set_title("Gradient magnitude across depth: the SAME chain-rule product,\ntwo very different outcomes")
ax.legend()
plt.show()

print(f"Sigmoid: gradient at output layer = {sigmoid_norms[-1]:.4f}, at input layer = {sigmoid_norms[0]:.2e}")
print(f"ReLU:    gradient at output layer = {relu_norms[-1]:.4f}, at input layer = {relu_norms[0]:.2e}")
print("\nBoth start from the SAME upstream gradient at the loss -- depth alone, compounded by")
print("the chain rule, is enough to shrink one to numerical zero and grow the other by 5 orders of magnitude.")

## 🐍 Implementation from Scratch

We implement forward AND backward passes for a dense layer and for self-attention -- by hand,
using only the formulas derived in the Theory section -- and verify every single gradient
against PyTorch's autograd. If our from-scratch derivation is correct, every number must match
exactly.

In [ ]:
import torch

# ---------- Dense layer: forward + backward, straight from the Theory section's formulas ----------
def dense_forward(x, W, b):
    z = x @ W + b
    return np.maximum(0, z), z  # ReLU

def dense_backward(dL_da, x, W, z):
    dL_dz = dL_da * (z > 0)     # chain through ReLU's local gradient
    dL_dW = np.outer(x, dL_dz)  # dL/dW = x . dL/dz^T
    dL_db = dL_dz
    dL_dx = dL_dz @ W.T          # becomes the UPSTREAM gradient for the previous layer
    return dL_dx, dL_dW, dL_db

rng = np.random.default_rng(0)
x0 = rng.standard_normal(4)
W0 = rng.standard_normal((4, 3)) * 0.5
b0 = rng.standard_normal(3) * 0.1
a0, z0 = dense_forward(x0, W0, b0)
dL_dx0, dL_dW0, dL_db0 = dense_backward(2 * a0, x0, W0, z0)  # dL/da for L = sum(a^2)

xt, Wt, bt = (torch.tensor(v, requires_grad=True) for v in (x0, W0, b0))
torch.sum(torch.relu(xt @ Wt + bt) ** 2).backward()
print("Dense layer backward, hand-derived vs. torch.autograd:")
print(f"  dL/dW match: {np.allclose(dL_dW0, Wt.grad.numpy())}")
print(f"  dL/db match: {np.allclose(dL_db0, bt.grad.numpy())}")
print(f"  dL/dx match: {np.allclose(dL_dx0, xt.grad.numpy())}")

# Pin down WHICH shape convention these formulas use (the Theory box states x in R^n,
# W in R^(n x m), z = x @ W -- confirm that's really what's in force here, and that the two
# equivalent ways of writing dL/dx (the boxed "W . dL/dz" vs. code's "dL_dz @ W.T") agree
# exactly, not just up to a transpose that happens to look right).
dL_dz0 = 2 * a0 * (z0 > 0)
print(f"\nShape check: x0 {x0.shape}, W0 {W0.shape}, z0 {z0.shape}  "
      f"(z = x @ W means W is (n_in, n_out), matching the Theory box's W in R^(n x m))")
print(f"dL/dW = outer(x, dL/dz) has shape {dL_dW0.shape}, same as W: {dL_dW0.shape == W0.shape}")
print(f"Theory's 'dL/dx = W . dL/dz' (W @ dL_dz) matches code's 'dL_dz @ W.T' exactly: "
      f"{np.allclose(W0 @ dL_dz0, dL_dz0 @ W0.T)}")

# ---------- Self-attention: forward + backward for Q, K, V ----------
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # log-sum-exp-style shift, previewed in 06.02
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention_forward(Q, K, V):
    scale = 1 / np.sqrt(Q.shape[-1])
    A = softmax(Q @ K.T * scale, axis=-1)
    return A @ V, A, scale

def attention_backward(dL_dO, A, Q, K, V, scale):
    dL_dA = dL_dO @ V.T
    dL_dV = A.T @ dL_dO
    # softmax's local gradient is a full per-row Jacobian, not elementwise -- this closed
    # form comes from differentiating softmax(scores)_i w.r.t. every score_j
    dL_dscores = A * (dL_dA - np.sum(A * dL_dA, axis=-1, keepdims=True))
    return (dL_dscores @ K) * scale, (dL_dscores.T @ Q) * scale, dL_dV

n, d_k = 4, 8
Q0, K0, V0 = (rng.standard_normal((n, d_k)) * 0.3 for _ in range(3))
O0, A0, scale0 = attention_forward(Q0, K0, V0)
dL_dQ0, dL_dK0, dL_dV0 = attention_backward(2 * O0, A0, Q0, K0, V0, scale0)

Qt, Kt, Vt = (torch.tensor(m, requires_grad=True) for m in (Q0, K0, V0))
At = torch.softmax(Qt @ Kt.T * scale0, dim=-1)
torch.sum((At @ Vt) ** 2).backward()
print("\nSelf-attention backward, hand-derived vs. torch.autograd:")
print(f"  dL/dQ match: {np.allclose(dL_dQ0, Qt.grad.numpy(), atol=1e-6)}")
print(f"  dL/dK match: {np.allclose(dL_dK0, Kt.grad.numpy(), atol=1e-6)}")
print(f"  dL/dV match: {np.allclose(dL_dV0, Vt.grad.numpy(), atol=1e-6)}")

## 🤖 Why This Matters for AI

Every equation above — dense backward, attention backward, gradient flow through depth — is
literally what happens inside `loss.backward()` for a real transformer, just scaled up. The
vanishing/exploding analysis explains a design choice in every modern architecture: the
**residual connection**, $\mathbf{y}=\mathbf{x}+f(\mathbf{x})$. Backpropagating through it
splits the upstream gradient down two paths — through $f$, and along the identity, whose local
gradient is exactly $1$ regardless of $f$. Below, we rebuild the vanishing-gradient experiment
with residual connections added to the sigmoid stack, and confirm the input-layer gradient no
longer collapses — proof, via the chain rule alone, of why residuals make deep nets trainable.

In [ ]:
# Does a residual connection actually rescue the vanishing gradient from the Visual Intuition
# cell? Rebuild the SAME sigmoid stack, but each layer now computes y = x + sigmoid(x @ W).
def gradient_norms_residual(n_layers, width=20, weight_scale=0.5):
    Ws = [rng.standard_normal((width, width)) * weight_scale for _ in range(n_layers)]
    x = rng.standard_normal(width)
    activs, zs = [x], []
    for W in Ws:
        z = activs[-1] @ W
        zs.append(z)
        activs.append(activs[-1] + sigmoid(z))   # RESIDUAL: y = x + sigmoid(x @ W)

    grad = np.ones(width)
    norms = [np.linalg.norm(grad)]
    for i in reversed(range(n_layers)):
        s = sigmoid(zs[i])
        local_grad_f = s * (1 - s)
        # dL/dx = dL/dy . (I + local_grad_f-weighted W^T) -- the "+1" from the identity
        # path is what keeps this from collapsing toward zero the way the plain stack did
        grad = grad + (grad * local_grad_f) @ Ws[i].T
        norms.append(np.linalg.norm(grad))
    return norms[::-1]

residual_norms = gradient_norms_residual(n_layers)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(sigmoid_norms, 'o-', color="#C44E52", label="plain sigmoid stack (vanishes)")
ax.semilogy(residual_norms, 'o-', color="#55A868", label="sigmoid stack + residual connections")
ax.set_xlabel("layer depth (0 = input side, 20 = output/loss side)")
ax.set_ylabel("||gradient|| (log scale)")
ax.set_title("Residual connections: same depth, same activation,\nno more vanishing gradient")
ax.legend()
plt.show()

print(f"Plain sigmoid stack:    gradient at input layer = {sigmoid_norms[0]:.2e}")
print(f"Residual sigmoid stack: gradient at input layer = {residual_norms[0]:.4f}")
print(f"\nThe identity path's local gradient is exactly 1 -- adding it back at every layer prevents")
print(f"the repeated multiplication by small numbers (sigmoid'(z) <= 0.25) from ever fully collapsing the signal.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a dense layer with $x=(1,-1)$, $W=\begin{pmatrix}0.5&-0.2\\0.3&0.8\end{pmatrix}$,
   $b=(0,0)$, ReLU, and $\partial L/\partial\mathbf{a}=(1,1)$, compute $\partial L/\partial W$
   by hand using this notebook's formulas. Verify with `dense_backward`, and as a structural
   sanity check confirm $\partial L/\partial W$ has the same shape as $W$ itself.

<details>
<summary>💡 Solution</summary>

$z=xW+b$: $z_0=(1)(0.5)+(-1)(0.3)=0.2$, $z_1=(1)(-0.2)+(-1)(0.8)=-1.0$, so $z=(0.2,-1.0)$ and
$a=\text{relu}(z)=(0.2,0)$. Backward: $\partial L/\partial z = \partial L/\partial a \odot
\mathbb{1}[z>0] = (1,1)\odot(1,0)=(1,0)$ — the second coordinate is exactly $0$ because
$z_1<0$ zeroed that branch of ReLU. Then $\partial L/\partial W = x^\top(\partial L/\partial z)$
(outer product of $x=(1,-1)$ with $(1,0)$):
$$\frac{\partial L}{\partial W} = \begin{pmatrix}1&0\\-1&0\end{pmatrix}.$$
`dense_backward` reproduces this exactly. $\partial L/\partial W$ is $2\times2$, matching $W$'s
own $2\times2$ shape — true by construction of the outer product ($\mathbf{x}\in\mathbb{R}^n$
against $\partial L/\partial\mathbf{z}\in\mathbb{R}^m$ always gives an $n\times m$ result), and
a check worth making explicitly since a transposed outer product would silently produce the
same *number* of entries but the wrong shape whenever $n\ne m$.

</details>

### 💭 UNDERSTAND
2. This notebook's dense layer uses ReLU, whose local gradient $\sigma'(z)$ is a single number
   per unit — so `dL_dz = dL_da * (z > 0)` is an elementwise product, and the activation's local
   "Jacobian" is effectively diagonal. Attention's softmax has no such elementwise shortcut
   (`attention_backward`'s `dL_dscores` line mixes every entry of a row together). Explain,
   structurally, why ReLU's Jacobian is diagonal while softmax's is dense, tracing the reason to
   how each function's $i$-th output depends on the *rest* of its input vector.

<details>
<summary>💡 Solution</summary>

$\text{relu}(z)_i = \max(0, z_i)$ depends on **only** $z_i$ — coordinate $i$ of the output is a
function of coordinate $i$ of the input alone, with zero dependence on any $z_j$, $j\ne i$. So
$\partial(\text{relu}(z))_i/\partial z_j = 0$ for every $i\ne j$, and the Jacobian is exactly
diagonal (confirmed numerically: a finite-difference Jacobian of ReLU has all off-diagonal
entries exactly $0$, matching $\mathbb{1}[z_i>0]$ on the diagonal). $\text{softmax}(z)_i =
e^{z_i}/\sum_k e^{z_k}$, by contrast, depends on **every** $z_k$ through the shared denominator
— increasing $z_j$ for any $j$ shrinks every other $s_i$ proportionally (probability mass is
conserved, so redistributing it necessarily couples every output to every input). This is
exactly why softmax's Jacobian $J=\text{diag}(\mathbf{s})-\mathbf{ss}^\top$ has nonzero
off-diagonal entries $-s_is_j$ for every pair $i\ne j$ (verified numerically: the same
finite-difference check on softmax gives clearly nonzero off-diagonal terms, unlike ReLU's).
The general principle: an **elementwise** function (ReLU, tanh, sigmoid) always has a diagonal
local Jacobian, letting backprop use a cheap elementwise product; any function whose outputs
*couple* through a shared normalization or reduction (softmax, LayerNorm) needs the full
matrix-vector product this notebook's `attention_backward` computes in closed form instead.

</details>

### ✏️ DERIVE
3. The Theory section states convolution's backward pass without derivation: "convolves the
   upstream gradient with the *flipped* kernel to get the input gradient, and convolves the
   input with the upstream gradient to get the kernel gradient." Derive this for 1D valid
   (no-padding) convolution: with $y_t=\sum_{i=0}^{K-1}w_ix_{t+i}$ for $t=0,\dots,T-1$
   ($T=N-K+1$), use the multivariate chain rule to write $\partial L/\partial x_n$ and
   $\partial L/\partial w_i$ as sums over every $y_t$ that each one feeds into, and confirm each
   sum matches the flipped-kernel/input-correlation description.

<details>
<summary>💡 Solution outline</summary>

Every $x_n$ feeds every $y_t$ for which $t\le n\le t+K-1$, i.e. $t\in\{n-K+1,\dots,n\}\cap[0,T-1]$
— a **fan-out** (`02.04`'s sum-over-paths rule applies directly). Since
$\partial y_t/\partial x_{t+i}=w_i$:
$$\frac{\partial L}{\partial x_n} = \sum_{t:\,0\le n-t\le K-1}\frac{\partial L}{\partial y_t}\,
w_{n-t}.$$
Substituting $i=n-t$ (so $t=n-i$) turns this into $\sum_i (\partial L/\partial y_{n-i})\,w_i$ —
exactly a cross-correlation of $\partial L/\partial\mathbf{y}$ with $\mathbf{w}$ evaluated at a
*reversed* offset $-i$ rather than $+i$, which is precisely convolving with the flipped kernel
$w_{K-1-i}$ after re-indexing (each output position pulls from upstream-gradient positions
running backward instead of forward). For the weight gradient, $w_i$ feeds every $y_t$ (it's
used identically at every sliding-window position — a fan-out over *all* $T$ positions, not a
local window), so:
$$\frac{\partial L}{\partial w_i} = \sum_{t=0}^{T-1}\frac{\partial L}{\partial y_t}\,x_{t+i}
= \sum_{t=0}^{T-1}\frac{\partial L}{\partial y_t}\,x_{t+i},$$
which is exactly a valid (no-padding) cross-correlation of $\mathbf{x}$ with
$\partial L/\partial\mathbf{y}$ playing the role of the sliding kernel — the input correlated
with the upstream gradient, no flip needed. Both identities check out numerically on
$x=(1,2,3,4)$, $w=(1,-1)$ (giving $y=(-1,-1,-1)$) with $\partial L/\partial\mathbf{y}=(1,1,1)$:
the derived formulas give $\partial L/\partial\mathbf{x}=(1,0,0,-1)$ and $\partial L/\partial
\mathbf{w}=(6,9)$, matching `torch.nn.functional.conv1d`'s autograd exactly.

</details>

### 🐍 IMPLEMENT
4. Implement `conv1d_forward(x, w)` and `conv1d_backward(dL_dy, x, w)` for valid (no-padding),
   stride-1 1D convolution, using the formulas from DERIVE (a direct sliding-window
   implementation is fine — no need for actual flipped-kernel convolution calls). Check both
   the forward output and every backward gradient against `torch.nn.functional.conv1d` and its
   autograd, on a random `x`, `w`, and upstream gradient.

<details>
<summary>✅ How to know you're right</summary>

Both `dL_dx` and `dL_dw` should match `torch.nn.functional.conv1d`'s autograd-computed gradients
exactly (`np.allclose` at default tolerance) — not approximately, since this is exact linear
algebra with no iterative approximation involved anywhere. Test on more than one random
`(x, w, dL_dy)` triple, since a bug in the sliding-window index bookkeeping (an off-by-one in
where `dL_dy[t]` scatters back into `dL_dx`) can easily produce correct results for a
symmetric or small test case while being wrong in general — a length-6 input with a length-3
kernel (giving 4 valid output positions) is large enough to catch most such bugs. If `dL_dw`
matches but `dL_dx` doesn't (or vice versa), the bug is isolated to just one of the two backward
formulas, which narrows the search considerably.

</details>

### 🤖 APPLY
5. Combine two ideas from this notebook — self-attention's backward pass and the residual
   connection from "Why This Matters for AI" — into one composed computation:
   $\mathbf{Y}=\mathbf{X}+\text{Attention}(\mathbf{X},\mathbf{X},\mathbf{X})$ (self-attention with
   a residual, the actual building block a transformer layer's attention sub-block uses), with
   loss $L=\sum(\mathbf{Y}^2)$. Derive and implement $\partial L/\partial\mathbf{X}$ by hand,
   noting that $\mathbf{X}$ now has **four** distinct paths to the loss (as $Q$, $K$, $V$, and
   via the direct residual identity path) whose contributions must all be summed. Check against
   `torch.autograd` on the identical computation.

<details>
<summary>✅ What you should observe</summary>

Your hand-composed $\partial L/\partial\mathbf{X}$ (summing the residual path's direct
contribution $2\mathbf{Y}$ with `attention_backward`'s three returned gradients $dQ+dK+dV$,
since $\mathbf{X}$ plays all three roles simultaneously in self-attention) should match
`torch.autograd`'s gradient on the same `Y = X + softmax(X @ X.T / sqrt(d_k)) @ X` computation
to within about $10^{-6}$ or tighter — this is exact backprop, not an approximation, so any
larger disagreement means a term was dropped or double-counted. The most common way to get this
wrong is forgetting one of the four contributions (most often the direct residual term, since
it's easy to mentally file "the gradient" as belonging only to the attention sub-block) — if
your answer differs from `torch.autograd`'s by *exactly* the value of $2\mathbf{Y}$, that is the
tell.

</details>

### 🚀 CHALLENGE
6. Extend the Visual Intuition section's gradient-flow-through-depth experiment from stacked
   dense layers to a stack of $n$ **self-attention** blocks with residual connections
   ($\mathbf{X}_{i+1}=\mathbf{X}_i+\text{Attention}(\mathbf{X}_i,\mathbf{X}_i,\mathbf{X}_i)$,
   no learned projections needed — reuse $\mathbf{X}_i$ directly as $Q,K,V$), and track the
   input-layer gradient norm as $n$ grows. You should find residual connections alone do *not*
   reproduce the dense-layer story: rather than staying stable, the input-layer gradient norm
   **grows** rapidly with depth. Diagnose the mechanism (it is not the same mechanism as ReLU's
   own exploding-gradient case), then add LayerNorm after each residual sum
   ($\mathbf{X}_{i+1}=\text{LayerNorm}(\mathbf{X}_i+\text{Attention}(\dots))$, the actual
   transformer-block recipe) and confirm this stabilizes the gradient norm across depth.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the growth numerically: with residual connections but no
normalization, the input-layer gradient norm grows roughly geometrically with depth — in one
run (a fixed random seed, width 8, sequence length 4), it went from $\approx170$ at $5$ layers
to $\approx5{,}600$ at $10$ layers to $\approx5.8\times10^6$ at $20$ layers, a growth factor of
roughly $2\times$ per added layer — while the *same* stack **without** residuals stayed bounded
(single digits) across all three depths. A strong answer correctly identifies the mechanism as
distinct from the sigmoid/ReLU case: there, exploding or vanishing gradients come from
repeatedly *multiplying* by a local Jacobian whose singular values are consistently above or
below $1$; here, each residual layer's backward pass **adds** the identity path's gradient to
the attention sub-block's own (generally nonzero, and not small) contribution at every one of
$n$ layers, so the gradient accumulates addends rather than shrinking or growing through
repeated multiplication — with no normalization anywhere to rescale it back down, that
accumulation compounds with depth. Adding LayerNorm after each residual sum (using the standard
LayerNorm backward formula, verified against finite differences first) should keep the
input-layer gradient norm in the same rough range as the output-layer's, regardless of depth —
in the same setup, LayerNorm keeps the input-layer norm within roughly $\pm50\%$ of the
output-layer norm at $5$, $10$, and $20$ layers, in sharp contrast to the unnormalized version's
many-orders-of-magnitude drift. A strong answer connects this to why every real transformer
architecture places a normalization layer at every single residual junction, never leaving
several un-normalized residual attention blocks stacked back-to-back — unlike a CNN or MLP,
where residual connections alone (ResNet's original design) are often enough.

</details>

---

## 📚 Further Reading
- Rumelhart, Hinton & Williams, ["Learning representations by back-propagating errors"](https://www.nature.com/articles/323533a0) (1986)
- He et al., ["Deep Residual Learning for Image Recognition"](https://arxiv.org/abs/1512.03385) (ResNet)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)
- Karpathy, [micrograd](https://github.com/karpathy/micrograd) (revisit alongside `02.04`'s `Value` engine)

---

*Next notebook: [Probability Fundamentals](../03_Probability_and_Statistics/01_probability_fundamentals.ipynb)*